# Chapter 7 — Secure-Agent Maturity: Course Capstone

## Objective

Score an agentic-AI program across six domains and leave with a maturity level, the weakest gap to close first, and a concrete next step.

By the end you will be able to:

- Score an agentic-AI program across six domains
- Explain how the governance gate caps overall maturity
- Distinguish the weakest domain from the gate
- Turn the assessment into a concrete next step
- Explain why the assessment is a recurring cadence

## Summary

This capstone synthesizes the whole course into one number. It scores an agentic-AI program across six domains — architecture, risk tooling, red-team readiness, autonomous defense, governance, and future-readiness — and returns an overall maturity level, the weakest domain to fix first, and a concrete next step. Governance is weighted heaviest and acts as a gate: with no audit trail, your maturity is capped no matter how strong everything else is. It is a small, standard-library tool you can re-run each quarter to track whether your posture is actually improving.

## Outline

**Chapter arc:** what you built → key takeaways → the throughline → a secure reference architecture → certifications → your next 90 days

This notebook covers:

- The six domains of a secure-agent program
- The maturity score and the governance gate
- Finding the weakest domain
- From score to next step
- Runnable demos
- A self-check quiz

## The big picture

This capstone synthesizes the course into a single, repeatable measurement. The arc is **Understand → Build → Attack → Defend → Govern**, and its throughline is that every attack has a defense and a governance control.

- **Six domains:** architecture, risk tooling, red-team readiness, autonomous defense, governance, and future-readiness.
- **The governance gate:** governance is weighted heaviest and caps overall maturity — no audit trail, no go-live, no matter how strong the rest.
- **The framework stack you now speak:** OWASP (ML + LLM), NIST AI RMF and AI 100-2, MITRE ATLAS, and Zero Trust.
- **A cadence, not a grade:** re-score each quarter and drive every domain toward 'managed', governance toward 'optimized'.

**The course arc**

<svg viewBox="0 0 984 124" width="100%" style="max-width:984px;height:auto;font-family:system-ui,-apple-system,Segoe UI,Arial,sans-serif" xmlns="http://www.w3.org/2000/svg" role="img"><defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs><rect width="984" height="124" rx="12" fill="#ffffff"/><rect x="22" y="22" width="156" height="62" rx="9" fill="#1d4ed8"/><text x="100" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Understand</text><line x1="178" y1="53" x2="218" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="218" y="22" width="156" height="62" rx="9" fill="#0f766e"/><text x="296" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Build</text><line x1="374" y1="53" x2="414" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="414" y="22" width="156" height="62" rx="9" fill="#b91c1c"/><text x="492" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Attack</text><line x1="570" y1="53" x2="610" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="610" y="22" width="156" height="62" rx="9" fill="#7c3aed"/><text x="688" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Defend</text><line x1="766" y1="53" x2="806" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="806" y="22" width="156" height="62" rx="9" fill="#a16207"/><text x="884" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Govern</text><text x="492" y="106" fill="#334155" font-size="12.5" text-anchor="middle">The throughline: every attack has a defense and a governance control — and no audit trail means no go-live.</text></svg>

## Setup

**Pure Python standard library** — no installs, no API key, no network. Everything runs as-is.

Imports used across the examples:

In [1]:
from __future__ import annotations
from dataclasses import dataclass

## Core building blocks

Each function below is introduced, then defined. Run them all, then see them in action in **Try it** and the **Live demo**.

### The six domains of a secure-agent program

The six course domains, each carrying a recommended next step and a certification to pursue.

> **Security lens:** the six domains are the whole course — each maps back to a chapter's arc from understand to govern.

In [2]:
@dataclass(frozen=True)
class Domain:
    key: str
    name: str
    chapter: str
    next_step: str
    cert: str


DOMAINS = [
    Domain("architecture", "Architecture & AI literacy", "Ch1",
           "Map your agents' four-layer attack surface (data/model/inference/runtime)",
           "CompTIA Security+"),
    Domain("risk", "GenAI risk & SecOps toolkit", "Ch2",
           "Stand up RAG grounding + DLP + a NIST AI RMF risk register",
           "Google Cloud Pro Security Engineer"),
    Domain("redteam", "Adversarial / red-team readiness", "Ch3-4",
           "Run an OWASP + MITRE ATLAS red-team pass against your top agent",
           "EC-Council CEH / SANS SEC545 (GAIPS)"),
    Domain("defense", "Detection & autonomous response", "Ch5",
           "Add agent-driven detection with reversible-auto / human-gated actions",
           "ISC2 CISSP"),
    Domain("governance", "Governance & Zero Trust", "Ch6",
           "Enforce the audit-trail exit criterion + run the OWASP gov checklist",
           "ISACA AAISM / Microsoft SC-100"),
    Domain("future", "Post-quantum & future readiness", "Ch6.5",
           "Inventory crypto, adopt crypto-agility toward FIPS 203/204/205",
           "Follow NIST PQC + AIRC"),
]

LEVELS = {0: "Absent", 1: "Initial", 2: "Developing", 3: "Managed", 4: "Optimized"}

### The maturity score and the governance gate

Average the six domain scores — governance weighted heaviest — into an overall 0–4 maturity level, then apply the governance gate.

> **Security lens:** governance is weighted heaviest and acts as a gate — no audit trail caps overall maturity at 'Initial' no matter how strong the rest.

In [3]:
def assess(scores: dict) -> dict:
    weights = {d.key: (1.5 if d.key == "governance" else 1.0) for d in DOMAINS}
    earned = sum(scores.get(d.key, 0) * weights[d.key] for d in DOMAINS)
    possible = sum(4 * weights[d.key] for d in DOMAINS)
    pct = round(100 * earned / possible)
    # weighted mean of the domain scores, rounded to the nearest 0..4 level
    level = min(4, round(earned / sum(weights.values())))
    # governance is a gate: no audit trail (gov score 0-1) caps you at "Initial"
    gated = scores.get("governance", 0) <= 1
    if gated:
        level = min(level, 1)
    return {"pct": pct, "level": level, "level_name": LEVELS[level],
            "governance_gated": gated}

### Finding the weakest domain

Return the single lowest-scoring domain: the gap to close first.

> **Security lens:** the weakest domain (biggest gap) and the gate (governance) are different things — closing the gap doesn't lift the level until the gate clears.

In [4]:
def weakest_domain(scores: dict) -> Domain:
    return min(DOMAINS, key=lambda d: scores.get(d.key, 0))

### From score to next step

Turn the weakest domain into a concrete next step and a certification to pursue.

> **Security lens:** the output is your assignment — a concrete next step and a certification for the weakest domain; drop it onto a 30/60/90-day plan.

In [5]:
def recommend(scores: dict) -> dict:
    d = weakest_domain(scores)
    return {"domain": d.name, "chapter": d.chapter,
            "next_step": d.next_step, "certification": d.cert,
            "current_level": LEVELS[scores.get(d.key, 0)]}

## Try it

Run the self-contained demo — the examples that need no external service — and read the output.

In [6]:
# A team strong on building and attacking, weak on governance/future-proofing.
scores = {
    "architecture": 3, "risk": 3, "redteam": 3,
    "defense": 2, "governance": 1, "future": 0,
}

result = recommend(scores)
m = assess(scores)

print("=== Secure-Agent Maturity Self-Assessment ===")
print(f"Overall maturity: {m['level']}/4 ({m['level_name']}), {m['pct']}%")
if m["governance_gated"]:
    print("  ! Capped at 'Initial': governance/audit-trail is the gate "
          "(Zero Trust exit criterion, NIST SP 800-207).")
print("\nPer-domain:")
for d in DOMAINS:
    s = scores.get(d.key, 0)
    bar = "#" * s + "." * (4 - s)
    print(f"  [{bar}] {LEVELS[s]:11} {d.name}  ({d.chapter})")

print(f"\nClose this gap first: {result['domain']} "
      f"(now: {result['current_level']})")
print(f"  Next step:     {result['next_step']}")
print(f"  Certification: {result['certification']}")
print("\nRe-run this assessment quarterly; aim to raise every domain to "
      "'Managed' (3) and governance to 'Optimized' (4).")

=== Secure-Agent Maturity Self-Assessment ===
Overall maturity: 1/4 (Initial), 48%
  ! Capped at 'Initial': governance/audit-trail is the gate (Zero Trust exit criterion, NIST SP 800-207).

Per-domain:
  [###.] Managed     Architecture & AI literacy  (Ch1)
  [###.] Managed     GenAI risk & SecOps toolkit  (Ch2)
  [###.] Managed     Adversarial / red-team readiness  (Ch3-4)
  [##..] Developing  Detection & autonomous response  (Ch5)
  [#...] Initial     Governance & Zero Trust  (Ch6)
  [....] Absent      Post-quantum & future readiness  (Ch6.5)

Close this gap first: Post-quantum & future readiness (now: Absent)
  Next step:     Inventory crypto, adopt crypto-agility toward FIPS 203/204/205
  Certification: Follow NIST PQC + AIRC

Re-run this assessment quarterly; aim to raise every domain to 'Managed' (3) and governance to 'Optimized' (4).


## Quiz

Answer each question, then fill in the `answers` dict in the next cell (`"A"`, `"B"`, …) and run it to grade yourself.

**1. Governance is weighted ×1.5 and acts as a gate. If it scores 0 or 1, overall maturity is capped at what?**

- A. Optimized
- B. Managed
- C. Initial
- D. Developing

**2. What does weakest_domain return?**

- A. The highest-scoring domain
- B. The single lowest-scoring domain — the gap to close first
- C. The average
- D. A random domain

**3. Raising post-quantum 0→3 does not lift the overall level while governance stays at 1. Why?**

- A. A bug
- B. The governance gate caps overall maturity until governance reaches 2+
- C. Wrong weights
- D. It does lift it

**4. The 0–4 maturity scale parallels which idea?**

- A. Random scoring
- B. Frameworks that ramp from ad-hoc to optimized/continuous (e.g., NIST AI RMF, CISA Zero Trust)
- C. Only financial risk
- D. CPU benchmarks

**5. recommend() turns the weakest domain into what?**

- A. A penalty
- B. A concrete next step and a certification to pursue
- C. A deletion
- D. A refund


In [7]:
import base64
_KEY = {'q1': 'Qw==', 'q2': 'Qg==', 'q3': 'Qg==', 'q4': 'Qg==', 'q5': 'Qg=='}   # base64 so the answers aren't obvious at a glance
_WHY = {'q1': "No audit trail caps you at 'Initial' (level 1).", 'q2': 'It surfaces the biggest gap.', 'q3': 'The gate overrides the weighted average until governance clears it.', 'q4': 'Both ramp ad-hoc → optimized.', 'q5': 'It returns an actionable next step for the gap.'}

def check(answers):
    key = {q: base64.b64decode(v).decode() for q, v in _KEY.items()}
    if all(str(answers.get(q, '')).strip().upper() in ('', '?') for q in key):
        print('Fill in the answers dict above (A/B/C/D) and re-run to grade.')
        return
    score = 0
    for q in sorted(key):
        got = str(answers.get(q, '')).strip().upper()
        if got == key[q]:
            score += 1; print(f'\u2713 {q}: {got}')
        else:
            print(f'\u2717 {q}: you put {got or "\u2014"} \u2014 correct is {key[q]}. {_WHY[q]}')
    print(f'\nScore: {score}/{len(key)}')

In [8]:
answers = {
    "q1": "?",   # A / B / C / D
    "q2": "?",   # A / B / C / D
    "q3": "?",   # A / B / C / D
    "q4": "?",   # A / B / C / D
    "q5": "?",   # A / B / C / D
}
check(answers)

Fill in the answers dict above (A/B/C/D) and re-run to grade.


## Key takeaways

- Maturity is a weighted average of six domains, with governance weighted heaviest and acting as a gate.
- No audit trail (governance 0–1) caps overall maturity at 'Initial', no matter how strong the rest.
- Fix the weakest domain first, but the gate — governance — is what unlocks the next level.
- Re-run the assessment on a cadence; security posture is a trend to manage, not a one-time grade.